# XAUUSD M5 - Data Import & EDA

**Data source:** Dukascopy m5 OHLCV, originally fetched with the use of dukascopy-node app (credits to https://github.com/Leo4815162342) and my ETL pipeline (https://github.com/ady7ady7/cross-market-etl-pipeline) and then stored in DigitalOcean PostgreSQL

**Coverage:** TBD (as I'm fetching more data soon)

**Note on volume:** Dukascopy volume is tick volume (quote count), not real traded size. Spot XAUUSD has no centralized exchange, so true volume is unavailable. Volume here is a proxy for quoting activity.

In [2]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', 5432)}/{os.getenv('DB_NAME')}"
)

In [7]:
query = """
SELECT
    timestamp AT TIME ZONE 'America/New_York' AS et_time,
    open, 
    high, 
    low, 
    close, 
    volume,
    EXTRACT(DOW FROM timestamp AT TIME ZONE 'America/New_York') AS day_of_week,
    (timestamp AT TIME ZONE 'America/New_York')::date AS trade_date
FROM xauusd_m5_tradfi_ohlcv
ORDER BY timestamp
"""

df = pd.read_sql(query, engine)
df.head()

,et_time,open,high,low,close,volume,day_of_week,trade_date
0,2021-01-03 17:00:00,1904.998,1910.898,1903.288,1908.850,0.68,0.0,2021-01-03
1,2021-01-03 17:05:00,1908.878,1909.258,1907.618,1908.568,0.46,0.0,2021-01-03
2,2021-01-03 17:10:00,1908.518,1909.405,1907.665,1908.805,0.21,0.0,2021-01-03
3,2021-01-03 17:15:00,1908.785,1909.678,1907.978,1909.358,0.09,0.0,2021-01-03
4,2021-01-03 17:20:00,1909.368,1910.588,1909.005,1909.048,0.14,0.0,2021-01-03


In [13]:
print(f"Shape: {df.shape}")
df['day_of_week'] = df['day_of_week'].astype(int)
df['trade_date'] = pd.to_datetime(df['trade_date'])


print(df.dtypes)

print(f"\nDate range: {df['et_time'].min()} → {df['et_time'].max()}")
print(f"Unique trade dates: {df['trade_date'].nunique()}")

print("Null counts:")
print(df.isnull().sum())


Shape: (391955, 8)
et_time        datetime64[us]
open                  float64
high                  float64
low                   float64
close                 float64
volume                float64
day_of_week             int64
trade_date      datetime64[s]
dtype: object

Date range: 2021-01-03 17:00:00 → 2026-07-19 17:55:00
Unique trade dates: 1723
Null counts:
et_time        0
open           0
high           0
low            0
close          0
volume         0
day_of_week    0
trade_date     0
dtype: int64


In [14]:
expected_bars = pd.date_range(
    start=df['et_time'].min(),
    end=df['et_time'].max(),
    freq='5min'
)
missing_bars = expected_bars.difference(df['et_time'])
print(f"Expected 5-min bars: {len(expected_bars)}")
print(f"Actual bars: {len(df)}")
print(f"Missing bars: {len(missing_bars)}")

Expected 5-min bars: 582636
Actual bars: 391955
Missing bars: 190681


In [15]:
missing_df = pd.DataFrame({'et_time': missing_bars})
missing_df['day_of_week'] = missing_df['et_time'].dt.dayofweek  # 0=Mon, 6=Sun

print("Missing bars by day of week:")
print(missing_df['day_of_week'].value_counts().sort_index())

Missing bars by day of week:
day_of_week
0     5448
1     3911
2     4173
3     5022
4    31564
5    83232
6    57331
Name: count, dtype: int64


In [16]:
weekday_missing = missing_df[missing_df['day_of_week'] < 5].copy()
weekday_missing['hour'] = weekday_missing['et_time'].dt.hour

print("Missing weekday bars by hour:")
print(weekday_missing['hour'].value_counts().sort_index())

Missing weekday bars by hour:
hour
0       228
1       228
2       229
3       228
4       228
5       228
6       228
7       228
8       228
9       229
10      228
11      303
12      486
13      696
14      780
15    10572
16     9302
17     3651
18     3638
19     3636
20     3636
21     3636
22     3636
23     3636
Name: count, dtype: int64


All 190k missing bars are accounted for: 

- weekend closures (Sat full day, Sun before 18:00 ET), 
- the daily 17:00–18:00 ET settlement break, 
- and ~228 holiday closures per hour slot. 

There are no unexpected intraday gaps on trading days.
